In [35]:
from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd().resolve()

for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "data").exists() and (candidate / "notebooks").exists():
        PROJECT_ROOT = candidate
        break

TRACK_STITCHING_DIR = (
        PROJECT_ROOT
        / "data"
        / "sample"
        / "processed"
        / "stage_8_track_stitching"
)

tracks_path = TRACK_STITCHING_DIR / "tracks.csv"
segmentation_events_path = TRACK_STITCHING_DIR / "segmentation_events.csv"
merge_seed_priors_path = TRACK_STITCHING_DIR / "merge_seed_priors.csv"
merge_track_links_path = TRACK_STITCHING_DIR / "merge_track_links.csv"

tracks = pd.read_csv(tracks_path)
segmentation_events = pd.read_csv(segmentation_events_path)
merge_seed_priors = pd.read_csv(merge_seed_priors_path)
merge_track_links = pd.read_csv(merge_track_links_path)

print("tracks:", tracks.shape)
print("segmentation_events:", segmentation_events.shape)
print("merge_seed_priors:", merge_seed_priors.shape)
print("merge_track_links:", merge_track_links.shape)

tracks: (4223, 7)
segmentation_events: (163, 37)
merge_seed_priors: (326, 12)
merge_track_links: (326, 6)


In [36]:
display_columns = [
    "event_id",
    "frame",
    "merged_frame",
    "track_a",
    "track_b",
    "merged_track",
    "merged_cell_id",
    "score",
]

segmentation_events[display_columns].sort_values(
    ["merged_frame", "event_id"]
).reset_index(drop=True)

,event_id,frame,merged_frame,track_a,track_b,merged_track,merged_cell_id,score
0,0,0,1,73,77,73,78,0.676798
1,1,0,1,26,30,30,36,0.664984
2,2,0,1,2,12,203,3,0.649314
3,3,0,1,178,193,178,185,0.623431
4,4,1,2,147,154,270,153,0.809047
...,...,...,...,...,...,...,...,...
158,158,18,19,156,925,156,208,0.662792
159,159,18,19,461,922,461,207,0.641967
160,160,18,19,903,486,486,46,0.628366
161,161,18,19,524,905,524,73,0.609915


In [37]:
EVENT_ID = 120

event_row = segmentation_events.loc[
    segmentation_events["event_id"] == EVENT_ID
    ].iloc[0]

event_row

event_id                       120
type                         merge
status                  hypothesis
frame                           12
merged_frame                    13
track_a                        414
track_b                        688
merged_track                   414
track_a_ended                False
track_b_ended                 True
merged_track_started         False
cell_a                          43
cell_b                         208
merged_cell                     40
cell_id_a                       44
cell_id_b                      209
merged_cell_id                  41
predicted_a_z            11.208945
predicted_a_y           201.531321
predicted_a_x             87.42714
predicted_b_z                  4.0
predicted_b_y           198.714286
predicted_b_x                 77.0
volume_a                     728.0
volume_b                       7.0
merged_volume                726.0
intensity_sum_a           789901.0
intensity_sum_b             4797.0
merged_intensity_sum

In [38]:
event_id = int(event_row["event_id"])
parent_track_a = int(event_row["track_a"])
parent_track_b = int(event_row["track_b"])
merged_track = int(event_row["merged_track"])
merged_frame = int(event_row["merged_frame"])
merged_cell_id = int(event_row["merged_cell_id"])

print("event_id:", event_id)
print("parent_track_a:", parent_track_a)
print("parent_track_b:", parent_track_b)
print("merged_track:", merged_track)
print("merged_frame:", merged_frame)
print("merged_cell_id:", merged_cell_id)

event_id: 120
parent_track_a: 414
parent_track_b: 688
merged_track: 414
merged_frame: 13
merged_cell_id: 41


In [39]:
VOXEL_SIZE = (1.625, 0.40625, 0.40625)

# ------------------------------------------------------------
# Parent histories up to the merge event
# ------------------------------------------------------------

parent_a_history = tracks[
    (tracks["track_id"] == parent_track_a)
    & (tracks["frame"] <= merged_frame)
    ].sort_values("frame").copy()

parent_b_history = tracks[
    (tracks["track_id"] == parent_track_b)
    & (tracks["frame"] <= merged_frame)
    ].sort_values("frame").copy()

# ------------------------------------------------------------
# Observed merged track from the merge frame onward
# ------------------------------------------------------------

merged_track_history = tracks[
    (tracks["track_id"] == merged_track)
    & (tracks["frame"] >= merged_frame)
    ].sort_values("frame").copy()

# ------------------------------------------------------------
# Predicted hidden centers
# ------------------------------------------------------------

event_priors = merge_seed_priors[
    merge_seed_priors["event_id"] == event_id
    ].copy()

# ------------------------------------------------------------
# Observed merged detection at the first merged frame
# ------------------------------------------------------------

merged_detection = tracks[
    (tracks["track_id"] == merged_track)
    & (tracks["frame"] == merged_frame)
    ].copy()

print("parent_a_history:", parent_a_history.shape)
print("parent_b_history:", parent_b_history.shape)
print("merged_track_history:", merged_track_history.shape)
print("event_priors:", event_priors.shape)
print("merged_detection:", merged_detection.shape)

parent_a_history: (8, 7)
parent_b_history: (1, 7)
merged_track_history: (7, 7)
event_priors: (2, 12)
merged_detection: (1, 7)


In [40]:
def build_track_array(
        df: pd.DataFrame,
        visual_track_id: int,
) -> np.ndarray:
    return np.column_stack(
        [
            np.full(len(df), visual_track_id, dtype=float),
            df["frame"].to_numpy(dtype=float),
            df["z"].to_numpy(dtype=float),
            df["y"].to_numpy(dtype=float),
            df["x"].to_numpy(dtype=float),
        ]
    )


track_arrays = []

if not parent_a_history.empty:
    track_arrays.append(
        build_track_array(parent_a_history, visual_track_id=1)
    )

if not parent_b_history.empty:
    track_arrays.append(
        build_track_array(parent_b_history, visual_track_id=2)
    )

if not merged_track_history.empty:
    track_arrays.append(
        build_track_array(merged_track_history, visual_track_id=3)
    )

merge_tracks_array = (
    np.vstack(track_arrays)
    if len(track_arrays) > 0
    else np.empty((0, 5), dtype=float)
)


predicted_center_points = event_priors[
    ["merged_frame", "predicted_z", "predicted_y", "predicted_x"]
].to_numpy(dtype=float)

predicted_center_properties = {
    "event_id": event_priors["event_id"].astype(int).to_numpy(),
    "parent_track_id": event_priors["parent_track_id"].astype(int).to_numpy(),
    "role": event_priors["role"].astype(str).to_numpy(),
    "seed_radius_um": event_priors["seed_radius_um"].astype(float).to_numpy(),
}

merged_detection_points = merged_detection[
    ["frame", "z", "y", "x"]
].to_numpy(dtype=float)

merged_detection_properties = {
    "track_id": merged_detection["track_id"].astype(int).to_numpy(),
    "cell_id": np.array([merged_cell_id] * len(merged_detection_points)),
    "event_id": np.array([event_id] * len(merged_detection_points)),
}

In [41]:
import zarr

SAMPLE_ID = "44b6_0113de3b"

ARRAY_PATH = (
        PROJECT_ROOT
        / "data"
        / "sample"
        / "biohub_5samples_20timepoints"
        / "train"
        / SAMPLE_ID
        / f"{SAMPLE_ID}.zarr"
        / "0"
)

raw_volume = zarr.open_array(str(ARRAY_PATH), mode="r")

print(raw_volume.shape)

(20, 64, 256, 256)


In [42]:
%gui qt

import napari

viewer = napari.Viewer(ndisplay=3)

# ------------------------------------------------------------
# Raw volume
# ------------------------------------------------------------

raw_layer = viewer.add_image(
    raw_volume,
    name="Raw Volume",
    scale=(1, *VOXEL_SIZE),
    rendering="mip",
    colormap="gray",
    opacity=1.0,
)

# ------------------------------------------------------------
# Merge-related tracks
# 1 = parent A
# 2 = parent B
# 3 = observed merged track
# ------------------------------------------------------------

merge_tracks_layer = viewer.add_tracks(
    merge_tracks_array,
    name="Merge Tracks",
    scale=(1, *VOXEL_SIZE),
    tail_length=20,
)

# ------------------------------------------------------------
# Predicted hidden centers
# ------------------------------------------------------------

predicted_centers_layer = viewer.add_points(
    predicted_center_points,
    name="Predicted Hidden Centers",
    scale=(1, *VOXEL_SIZE),
    size=6,
    face_color="cyan",
    border_color="white",
    properties=predicted_center_properties,
    text={
        "string": "P{parent_track_id}",
        "size": 10,
        "color": "white",
        "anchor": "center",
    },
)

# ------------------------------------------------------------
# Observed merged detection
# ------------------------------------------------------------

merged_detection_layer = viewer.add_points(
    merged_detection_points,
    name="Observed Merged Detection",
    scale=(1, *VOXEL_SIZE),
    size=8,
    face_color="yellow",
    border_color="red",
    properties=merged_detection_properties,
    text={
        "string": "M{track_id}",
        "size": 10,
        "color": "yellow",
        "anchor": "center",
    },
)

viewer.dims.set_point(0, merged_frame)

viewer.camera.angles = (45, 30, 135)
viewer.reset_view()